In [ ]:
# -----------------------------------------------------------------------------
# Imports
# -----------------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt

from dataclasses import dataclass, replace, fields
from pathlib import Path

plt.rcParams.update({"font.size": 12})


# Automatically reload edited modules during notebook development
%load_ext autoreload
%autoreload 2


# -----------------------------------------------------------------------------
# Project modules
# -----------------------------------------------------------------------------

from modules.simulation_io import (
    latest_saved_simulation_dir,
    load_saved_simulation,
    params_to_json_dict,
    json_safe_value,
    save_simulation_2d,
    save_static_problem_2d,
)

from modules.grid_kernel_confinement_io import (
    build_static_problem_2d,
)

from modules.hartree_io import (
    solve_self_consistent_2d_optimized,
    average_density_2d,
)

In [ ]:
@dataclass
class Params2D:
    # full rectangular Hall bar dimensions in nm
    Lx: float = 2000.0
    Ly: float = 1000.0
    Nx: int = 251
    Ny: int = 126

    use_elliptic_mask: bool = False

    # material / electrostatics
    eps_r: float = 13.0
    n_donor: float = 1.5e-3
    n_target: float = 1.0e-3
    n_surf: float = .5e-3

    dS: float = 62  # nm
    donor_margin_nm: float = 0.0
    donor_smooth_nm: float = 0.0
    
    donor_disorder_rms: float = 1.5e-3 * 0.2       # nm^-2
    donor_disorder_sigma_nm: float = 20.0 # correlation length [nm]
    donor_disorder_seed: int = 12345

    # top-gate distance in nm. Use None for bare Coulomb.
    dist_fixed: float = 1e6

    # Partial top-gate image configuration
    use_partial_top_gate: bool = True
    x_gate_edge_nm: float = 200.0

    # Donor image charges
    donor_extra_dist_nm: float = -90.0

    # magnetic field / LLs
    B: float = 0.75
    g_spin: float = 0.0
    n_LL: int = 10
    gamma_rel: float = 0.02
    use_B_scaling: bool = True

    # self-consistency
    max_iter: int = 10000
    mix: float = 0.0015
    tol: float = 8e-9
    smooth_sigma_nm: float = 0.0

    # Coulomb regularization
    diagonal_cutoff_nm: float | None = None

    @property
    def kappa0(self):
        return 1440.0 / self.eps_r

    @property
    def hbar_omega_c(self):
        return 1.76 * self.B

    @property
    def lB(self):
        return 25.658 / np.sqrt(self.B)

    @property
    def Gamma(self):
        if self.use_B_scaling:
            return (
                self.gamma_rel
                * self.hbar_omega_c
                * np.sqrt(10.0 / self.B)
            )
        return self.gamma_rel * self.hbar_omega_c


p = Params2D()

In [ ]:
# -----------------------------------------------------------------------------
# B sweep configuration
# -----------------------------------------------------------------------------
"""arr_u = np.linspace(0,1,200)
arr_u = arr_u**2

arr_B = .69 + (2-.69)*arr_u"""
arr_B = np.linspace(2,4,100)

RESULTS_DIR = Path("data_sweep")

USE_LATEST_SAVED_SIMULATION = True
USE_SAVED_U_AS_INITIAL_GUESS = True

SAVE_THIS_RUN = True

# True:
#   each run contains all static arrays and is completely standalone.
#
# False:
#   static electrostatic arrays are stored once in static_data.npz.
SAVE_STATIC_ARRAYS_EACH_RUN = True


# Parameters applied to every point of the sweep.
# B itself will be inserted automatically inside the loop.
FIXED_PARAMETER_OVERRIDES = {
    "tol": 1e-8, "n_LL":5,
}

In [ ]:
# -----------------------------------------------------------------------------
# Resume previous calculation
# -----------------------------------------------------------------------------

RESUMED_RUN_DIR = None
RESUMED_METADATA = None
RESUMED_DATA = None
U_INITIAL_FROM_SAVED_RUN = None


if USE_LATEST_SAVED_SIMULATION:

    RESUMED_RUN_DIR = latest_saved_simulation_dir(
        RESULTS_DIR
    )

    if RESUMED_RUN_DIR is None:

        print(
            f"No saved simulation found in {RESULTS_DIR}. "
            "Starting from current parameters."
        )

    else:

        RESUMED_METADATA, RESUMED_DATA = load_saved_simulation(
            RESUMED_RUN_DIR
        )

        saved_params = RESUMED_METADATA.get(
            "params",
            {},
        )

        valid_fields = {f.name for f in fields(Params2D)}

        saved_params = {
            k: v
            for k, v in saved_params.items()
            if k in valid_fields
        }

        p = replace(
            p,
            **saved_params,
        )

        print(
            f"Loaded previous simulation from:\n"
            f"{RESUMED_RUN_DIR}"
        )


# -----------------------------------------------------------------------------
# Apply fixed sweep parameters
# -----------------------------------------------------------------------------

valid_fields = {f.name for f in fields(Params2D)}

bad = sorted(
    set(FIXED_PARAMETER_OVERRIDES)
    - valid_fields
)

if bad:
    raise ValueError(
        "Unknown Params2D fields in "
        f"FIXED_PARAMETER_OVERRIDES: {bad}"
    )

p = replace(
    p,
    **FIXED_PARAMETER_OVERRIDES,
)


# -----------------------------------------------------------------------------
# Build all B-independent electrostatics once
# -----------------------------------------------------------------------------

print(
    "Building B-independent electrostatic problem "
    "and precomputing kernel FFTs..."
)

static = build_static_problem_2d(
    p,
    dist=p.dist_fixed,
)


# -----------------------------------------------------------------------------
# Optionally save static arrays once
# -----------------------------------------------------------------------------

if SAVE_STATIC_ARRAYS_EACH_RUN:

    STATIC_DATA_PATH = None

else:

    STATIC_DATA_PATH = save_static_problem_2d(
        static,
        RESULTS_DIR,
    )

    print(
        "Static arrays saved to:",
        STATIC_DATA_PATH,
    )


# -----------------------------------------------------------------------------
# Initial U for first sweep point
# -----------------------------------------------------------------------------

if (
    USE_LATEST_SAVED_SIMULATION
    and RESUMED_DATA is not None
    and USE_SAVED_U_AS_INITIAL_GUESS
):

    saved_U = RESUMED_DATA["U"]

    if saved_U.shape == (p.Ny, p.Nx):

        U_INITIAL_FROM_SAVED_RUN = saved_U.copy()

        print(
            "Using saved U as initial guess "
            "for the first B point."
        )

    else:

        print(
            "Saved U has incompatible grid shape. "
            "Starting from U_ext instead.\n"
            f"saved: {saved_U.shape}, "
            f"current: {(p.Ny, p.Nx)}"
        )


# We have copied everything required from the npz file.
if RESUMED_DATA is not None:
    RESUMED_DATA.close()
    RESUMED_DATA = None


previous_saved_run_dir = RESUMED_RUN_DIR
U_warm = U_INITIAL_FROM_SAVED_RUN

In [ ]:
# -----------------------------------------------------------------------------
# Run B sweep
# -----------------------------------------------------------------------------

for nB, Bs in enumerate(arr_B):

    PARAMETER_OVERRIDES = {
        "B": float(Bs),
        **FIXED_PARAMETER_OVERRIDES,
    }

    bad = sorted(
        set(PARAMETER_OVERRIDES)
        - valid_fields
    )

    if bad:
        raise ValueError(
            "Unknown Params2D fields in "
            f"PARAMETER_OVERRIDES: {bad}"
        )

    pB = replace(
        p,
        **PARAMETER_OVERRIDES,
    )

    print()
    print("=" * 80)
    print(
        f"B sweep {nB + 1}/{len(arr_B)} "
        f": B = {Bs:.8g} T"
    )

    print(
        f"lB = {pB.lB:.3f} nm, "
        f"hbar omega_c = {pB.hbar_omega_c:.3f} meV, "
        f"Gamma = {pB.Gamma:.4f} meV, "
        f"nu_avg = "
        f"{2*np.pi*pB.lB**2*pB.n_target:.4f}"
    )

    # ---------------------------------------------------------------------
    # Self-consistent Hartree solution
    # ---------------------------------------------------------------------

    res = solve_self_consistent_2d_optimized(
        pB,
        static,
        U_initial=U_warm,
        verbose=True,
    )
    

    # ---------------------------------------------------------------------
    # Diagnostics
    # ---------------------------------------------------------------------

    print("history tail:")
    print(res["history"][-5:])

    print(
        "average density:",
        average_density_2d(
            res["n"],
            res["mask"],
            res["dx"],
            res["dy"],
        ),
        "target:",
        pB.n_target,
    )

    print(
        "center filling:",
        res["nu"][
            pB.Ny // 2,
            pB.Nx // 2,
        ],
    )

    # ---------------------------------------------------------------------
    # Save
    # ---------------------------------------------------------------------

    if SAVE_THIS_RUN:

        SAVED_RUN_DIR = save_simulation_2d(
            res,
            pB,
            resumed_run_dir=previous_saved_run_dir,
            parameter_overrides=PARAMETER_OVERRIDES,
            results_dir=RESULTS_DIR,
            static_data_path=STATIC_DATA_PATH,
            save_static_arrays_each_run=SAVE_STATIC_ARRAYS_EACH_RUN,
        )

        previous_saved_run_dir = SAVED_RUN_DIR

    else:

        SAVED_RUN_DIR = None

    # ---------------------------------------------------------------------
    # Warm start next B point directly from RAM
    # ---------------------------------------------------------------------

    if USE_SAVED_U_AS_INITIAL_GUESS:
        U_warm = res["U"].copy()
    else:
        U_warm = None


print()
print("Sweep complete.")